# 09. Validation and Stability

## Objective

This notebook validates the selected WOE Logistic Regression scorecard model from Notebook 07.

- consolidate Train / Validation / OOT performance;
- validate AUROC, Gini, KS and Brier score stability;
- review PD and score distributions;
- calculate PSI for score and model variables;
- review performance by issue quarter;
- review calibration by sample and score band;


Inputs:

```text
data/processed/06.woe_binning_iv_analysis/
data/processed/07.logistic_regression_scorecard/
```

Outputs:

```text
data/processed/09.validation_and_stability/
```


In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Paths and input files

In [ ]:
PROJECT_ROOT = Path("..")

WOE_INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "06.woe_binning_iv_analysis"
MODEL_INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "07.logistic_regression_scorecard"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "09.validation_and_stability"
CHARTS_DIR = OUTPUT_DIR / "charts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_INPUT_DIR / "logistic_regression_model.pkl"
FEATURES_FILE = MODEL_INPUT_DIR / "final_model_features.pkl"

TRAIN_WOE_CANDIDATES = [
    WOE_INPUT_DIR / "train_woe_final.pkl",
    WOE_INPUT_DIR / "train_woe.pkl",
]
VALIDATION_WOE_CANDIDATES = [
    WOE_INPUT_DIR / "validation_woe_final.pkl",
    WOE_INPUT_DIR / "validation_woe.pkl",
]
OOT_WOE_CANDIDATES = [
    WOE_INPUT_DIR / "oot_test_woe_final.pkl",
    WOE_INPUT_DIR / "oot_test_woe.pkl",
]

TRAIN_PRED_FILE = MODEL_INPUT_DIR / "train_predictions.pkl"
VALIDATION_PRED_FILE = MODEL_INPUT_DIR / "validation_predictions.pkl"
OOT_PRED_FILE = MODEL_INPUT_DIR / "oot_predictions.pkl"


def first_existing_file(candidates, label):
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Missing expected {label}. Checked: "
        + ", ".join(str(p) for p in candidates)
    )


TRAIN_WOE_FILE = first_existing_file(TRAIN_WOE_CANDIDATES, "train WOE file")
VALIDATION_WOE_FILE = first_existing_file(VALIDATION_WOE_CANDIDATES, "validation WOE file")
OOT_WOE_FILE = first_existing_file(OOT_WOE_CANDIDATES, "OOT WOE file")

for path in [MODEL_FILE, FEATURES_FILE, TRAIN_PRED_FILE, VALIDATION_PRED_FILE, OOT_PRED_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected input file: {path}")

print(f"WOE input directory:   {WOE_INPUT_DIR.resolve()}")
print(f"Model input directory: {MODEL_INPUT_DIR.resolve()}")
print(f"Output directory:      {OUTPUT_DIR.resolve()}")
print(f"Charts directory:      {CHARTS_DIR.resolve()}")

print("\nUsing WOE files:")
print(f"- Train:      {TRAIN_WOE_FILE.name}")
print(f"- Validation: {VALIDATION_WOE_FILE.name}")
print(f"- OOT:        {OOT_WOE_FILE.name}")

## 2. Load model, features, WOE datasets and predictions

In [ ]:
with open(MODEL_FILE, "rb") as f:
    model = pickle.load(f)

with open(FEATURES_FILE, "rb") as f:
    model_features = pickle.load(f)

train_woe = pd.read_pickle(TRAIN_WOE_FILE)
validation_woe = pd.read_pickle(VALIDATION_WOE_FILE)
oot_woe = pd.read_pickle(OOT_WOE_FILE)

train_pred = pd.read_pickle(TRAIN_PRED_FILE)
validation_pred = pd.read_pickle(VALIDATION_PRED_FILE)
oot_pred = pd.read_pickle(OOT_PRED_FILE)

TARGET_COL = "target_bad"

# Normalize feature names defensively.
model_features = [
    c for c in model_features
    if c in train_woe.columns and c != TARGET_COL
]

if len(model_features) == 0:
    raise ValueError("No model features found in WOE datasets.")

print(f"Model features: {len(model_features)}")
print(f"Train WOE shape:      {train_woe.shape}")
print(f"Validation WOE shape: {validation_woe.shape}")
print(f"OOT WOE shape:        {oot_woe.shape}")

model_features

## 3. Ensure predictions are available and aligned

The notebook uses the predictions generated in Notebook 07.  
If needed, it can also recalculate predictions from the saved model.


In [ ]:
def ensure_predictions(data_woe, pred_df, sample_name):
    out = pred_df.copy()

    if "pd_pred" not in out.columns:
        X = data_woe[model_features].fillna(0)
        out["pd_pred"] = model.predict_proba(X)[:, 1]

    if "score" not in out.columns:
        # Recreate score using same standard parameters as Notebook 07
        BASE_SCORE = 600
        BASE_ODDS = 20
        PDO = 50
        factor = PDO / np.log(2)
        offset = BASE_SCORE - factor * np.log(BASE_ODDS)
        pd_values = np.clip(out["pd_pred"].values, 1e-6, 1 - 1e-6)
        odds = (1 - pd_values) / pd_values
        out["score"] = offset + factor * np.log(odds)

    if TARGET_COL not in out.columns:
        out[TARGET_COL] = data_woe[TARGET_COL].values

    out["sample"] = sample_name

    for audit_col in ["issue_d", "issue_quarter", "issue_month"]:
        if audit_col not in out.columns and audit_col in data_woe.columns:
            out[audit_col] = data_woe[audit_col].values

    return out


train_pred = ensure_predictions(train_woe, train_pred, "train")
validation_pred = ensure_predictions(validation_woe, validation_pred, "validation")
oot_pred = ensure_predictions(oot_woe, oot_pred, "oot_test")

predictions_all = pd.concat([train_pred, validation_pred, oot_pred], ignore_index=True)

predictions_all.head()

## 4. Performance helper functions

In [ ]:
def gini_from_auc(auc):
    return 2 * auc - 1


def ks_statistic(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    ks_values = tpr - fpr
    idx = np.argmax(ks_values)

    return {
        "ks": ks_values[idx],
        "threshold": thresholds[idx],
        "tpr": tpr[idx],
        "fpr": fpr[idx],
    }


def evaluate_sample(pred_df, sample_name):
    y_true = pred_df[TARGET_COL].astype(int)
    y_score = pred_df["pd_pred"]

    auc = roc_auc_score(y_true, y_score)
    ks = ks_statistic(y_true, y_score)

    return {
        "sample": sample_name,
        "rows": len(pred_df),
        "bad_rate": y_true.mean(),
        "avg_predicted_pd": y_score.mean(),
        "auc": auc,
        "gini": gini_from_auc(auc),
        "ks": ks["ks"],
        "ks_threshold": ks["threshold"],
        "brier_score": brier_score_loss(y_true, y_score),
        "avg_score": pred_df["score"].mean(),
        "min_score": pred_df["score"].min(),
        "max_score": pred_df["score"].max(),
    }


def plot_metric_by_sample(perf, metric, charts_dir):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(perf["sample"], perf[metric], alpha=0.8)
    ax.set_xlabel("Sample")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} by Sample")
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"{metric}_by_sample.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_distribution_by_sample(predictions, column, charts_dir, bins=40):
    fig, ax = plt.subplots(figsize=(9, 5))

    for sample in ["train", "validation", "oot_test"]:
        subset = predictions[predictions["sample"] == sample]
        ax.hist(subset[column], bins=bins, alpha=0.35, label=sample, density=True)

    ax.set_xlabel(column)
    ax.set_ylabel("Density")
    ax.set_title(f"{column} Distribution by Sample")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"{column}_distribution_by_sample.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. Train / Validation / OOT performance stability

In [ ]:
performance_summary = pd.DataFrame([
    evaluate_sample(train_pred, "train"),
    evaluate_sample(validation_pred, "validation"),
    evaluate_sample(oot_pred, "oot_test"),
])

performance_summary

In [ ]:
for metric in ["auc", "gini", "ks", "brier_score"]:
    plot_metric_by_sample(performance_summary, metric, CHARTS_DIR)

## 6. Performance drift versus train

This table compares validation and OOT performance against train performance.


In [ ]:
train_metrics = performance_summary[performance_summary["sample"] == "train"].iloc[0]

performance_drift_rows = []

for _, row in performance_summary.iterrows():
    sample = row["sample"]
    performance_drift_rows.append({
        "sample": sample,
        "auc": row["auc"],
        "auc_diff_vs_train": row["auc"] - train_metrics["auc"],
        "gini": row["gini"],
        "gini_diff_vs_train": row["gini"] - train_metrics["gini"],
        "ks": row["ks"],
        "ks_diff_vs_train": row["ks"] - train_metrics["ks"],
        "brier_score": row["brier_score"],
        "brier_diff_vs_train": row["brier_score"] - train_metrics["brier_score"],
    })

performance_drift = pd.DataFrame(performance_drift_rows)

performance_drift

## 7. PD and score distribution stability

In [ ]:
plot_distribution_by_sample(predictions_all, "pd_pred", CHARTS_DIR, bins=50)
plot_distribution_by_sample(predictions_all, "score", CHARTS_DIR, bins=50)

distribution_summary = (
    predictions_all
    .groupby("sample")
    .agg(
        rows=(TARGET_COL, "size"),
        bad_rate=(TARGET_COL, "mean"),
        avg_pd=("pd_pred", "mean"),
        p01_pd=("pd_pred", lambda x: np.quantile(x, 0.01)),
        p05_pd=("pd_pred", lambda x: np.quantile(x, 0.05)),
        p50_pd=("pd_pred", "median"),
        p95_pd=("pd_pred", lambda x: np.quantile(x, 0.95)),
        p99_pd=("pd_pred", lambda x: np.quantile(x, 0.99)),
        avg_score=("score", "mean"),
        p01_score=("score", lambda x: np.quantile(x, 0.01)),
        p05_score=("score", lambda x: np.quantile(x, 0.05)),
        p50_score=("score", "median"),
        p95_score=("score", lambda x: np.quantile(x, 0.95)),
        p99_score=("score", lambda x: np.quantile(x, 0.99)),
    )
    .reset_index()
)

distribution_summary

## 8. PSI helper functions

Population Stability Index interpretation used here:

| PSI | Interpretation |
|---:|---|
| < 0.10 | stable |
| 0.10 - 0.25 | moderate shift |
| > 0.25 | significant shift |

PSI is calculated using the train sample as the reference population.


In [ ]:
def psi_band(psi):
    if pd.isna(psi):
        return "not_available"
    if psi < 0.10:
        return "stable"
    if psi < 0.25:
        return "moderate_shift"
    return "significant_shift"


def create_numeric_bins(reference_series, bins=10):
    s = pd.to_numeric(reference_series, errors="coerce").dropna()

    if s.nunique() <= 2:
        return None

    try:
        _, edges = pd.qcut(s, q=bins, retbins=True, duplicates="drop")
    except Exception:
        _, edges = pd.cut(s, bins=bins, retbins=True, duplicates="drop")

    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    edges[0] = -np.inf
    edges[-1] = np.inf

    return edges


def calculate_psi(reference, comparison, bins=10, variable_name="variable"):
    ref = pd.Series(reference).copy()
    comp = pd.Series(comparison).copy()

    edges = create_numeric_bins(ref, bins=bins)

    if edges is not None:
        ref_bins = pd.cut(pd.to_numeric(ref, errors="coerce"), bins=edges, include_lowest=True).astype(str)
        comp_bins = pd.cut(pd.to_numeric(comp, errors="coerce"), bins=edges, include_lowest=True).astype(str)
    else:
        ref_bins = ref.astype("object").where(ref.notna(), "Missing").astype(str)
        comp_bins = comp.astype("object").where(comp.notna(), "Missing").astype(str)

    ref_bins = ref_bins.where(ref.notna(), "Missing")
    comp_bins = comp_bins.where(comp.notna(), "Missing")

    categories = sorted(set(ref_bins.unique()).union(set(comp_bins.unique())))

    rows = []
    eps = 1e-6

    for cat in categories:
        ref_pct = (ref_bins == cat).mean()
        comp_pct = (comp_bins == cat).mean()
        psi_component = (comp_pct - ref_pct) * np.log((comp_pct + eps) / (ref_pct + eps))

        rows.append({
            "variable": variable_name,
            "bin": cat,
            "reference_pct": ref_pct,
            "comparison_pct": comp_pct,
            "psi_component": psi_component,
        })

    detail = pd.DataFrame(rows)
    psi_value = detail["psi_component"].sum()

    return psi_value, detail

## 9. Score and PD PSI

In [ ]:
score_psi_rows = []
score_psi_details = []

for variable in ["pd_pred", "score"]:
    for sample_name, comp_df in {
        "validation": validation_pred,
        "oot_test": oot_pred,
    }.items():
        psi_value, detail = calculate_psi(
            train_pred[variable],
            comp_df[variable],
            bins=10,
            variable_name=variable,
        )

        score_psi_rows.append({
            "variable": variable,
            "comparison_sample": sample_name,
            "psi": psi_value,
            "psi_band": psi_band(psi_value),
        })

        detail.insert(1, "comparison_sample", sample_name)
        score_psi_details.append(detail)

score_psi_summary = pd.DataFrame(score_psi_rows)
score_psi_detail = pd.concat(score_psi_details, ignore_index=True)

score_psi_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = score_psi_summary.copy()
plot_data["label"] = plot_data["variable"] + " | " + plot_data["comparison_sample"]
ax.barh(plot_data["label"], plot_data["psi"], alpha=0.8)
ax.axvline(0.10, linestyle="--", label="0.10")
ax.axvline(0.25, linestyle="--", label="0.25")
ax.set_xlabel("PSI")
ax.set_ylabel("Variable / sample")
ax.set_title("Score and PD PSI vs Train")
ax.legend()
ax.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "score_pd_psi.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Characteristic stability: PSI for final WOE variables

In [ ]:
feature_psi_rows = []
feature_psi_details = []

for feature in model_features:
    for sample_name, comp_df in {
        "validation": validation_woe,
        "oot_test": oot_woe,
    }.items():
        psi_value, detail = calculate_psi(
            train_woe[feature],
            comp_df[feature],
            bins=10,
            variable_name=feature,
        )

        feature_psi_rows.append({
            "variable": feature,
            "comparison_sample": sample_name,
            "psi": psi_value,
            "psi_band": psi_band(psi_value),
        })

        detail.insert(1, "comparison_sample", sample_name)
        feature_psi_details.append(detail)

feature_psi_summary = pd.DataFrame(feature_psi_rows).sort_values(
    ["comparison_sample", "psi"],
    ascending=[True, False],
)

feature_psi_detail = pd.concat(feature_psi_details, ignore_index=True)

feature_psi_summary

In [ ]:
for sample_name in ["validation", "oot_test"]:
    data = feature_psi_summary[feature_psi_summary["comparison_sample"] == sample_name].copy()
    data = data.sort_values("psi", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(5, len(data) * 0.35)))
    ax.barh(data["variable"], data["psi"], alpha=0.8)
    ax.axvline(0.10, linestyle="--", label="0.10")
    ax.axvline(0.25, linestyle="--", label="0.25")
    ax.set_xlabel("PSI")
    ax.set_ylabel("WOE feature")
    ax.set_title(f"Final Model Feature PSI vs Train - {sample_name}")
    ax.legend()
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / f"feature_psi_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 11. Calibration by sample

The model is calibrated well when average predicted PD is close to the observed bad rate.

This section checks calibration at total sample level and by risk decile.


In [ ]:
calibration_summary = (
    predictions_all
    .groupby("sample")
    .agg(
        accounts=(TARGET_COL, "size"),
        observed_bad_rate=(TARGET_COL, "mean"),
        avg_predicted_pd=("pd_pred", "mean"),
    )
    .reset_index()
)

calibration_summary["calibration_gap"] = (
    calibration_summary["avg_predicted_pd"] 
    - calibration_summary["observed_bad_rate"]
)

calibration_summary

In [ ]:
def calibration_by_decile(pred_df, sample_name):
    temp = pred_df.copy()
    temp["risk_decile"] = pd.qcut(
        temp["pd_pred"].rank(method="first", ascending=False),
        q=10,
        labels=list(range(1, 11)),
    ).astype(int)

    out = (
        temp.groupby("risk_decile")
        .agg(
            accounts=(TARGET_COL, "size"),
            observed_bad_rate=(TARGET_COL, "mean"),
            avg_predicted_pd=("pd_pred", "mean"),
            min_pd=("pd_pred", "min"),
            max_pd=("pd_pred", "max"),
            avg_score=("score", "mean"),
        )
        .reset_index()
    )

    out.insert(0, "sample", sample_name)
    out["calibration_gap"] = out["avg_predicted_pd"] - out["observed_bad_rate"]
    return out


calibration_deciles = pd.concat([
    calibration_by_decile(train_pred, "train"),
    calibration_by_decile(validation_pred, "validation"),
    calibration_by_decile(oot_pred, "oot_test"),
], ignore_index=True)

calibration_deciles

In [ ]:
for sample_name in ["train", "validation", "oot_test"]:
    data = calibration_deciles[calibration_deciles["sample"] == sample_name]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(data["risk_decile"].astype(str), data["observed_bad_rate"], alpha=0.75, label="Observed bad rate")
    ax.plot(
        data["risk_decile"].astype(str),
        data["avg_predicted_pd"],
        color="red",
        marker="o",
        linewidth=2.5,
        label="Avg predicted PD",
    )
    ax.set_xlabel("Risk decile: 1 = highest risk")
    ax.set_ylabel("Rate")
    ax.set_title(f"Calibration by Decile - {sample_name}")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / f"calibration_deciles_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 12. Performance by issue quarter

- bad rate by vintage;
- average predicted PD by vintage;
- average score by vintage;
- AUROC by vintage where enough bad/good observations exist.


In [ ]:
def safe_auc(group):
    if group[TARGET_COL].nunique() < 2:
        return np.nan
    return roc_auc_score(group[TARGET_COL], group["pd_pred"])


if "issue_quarter" in predictions_all.columns:
    quarterly_performance = (
        predictions_all
        .groupby(["sample", "issue_quarter"])
        .apply(lambda g: pd.Series({
            "accounts": len(g),
            "bads": g[TARGET_COL].sum(),
            "observed_bad_rate": g[TARGET_COL].mean(),
            "avg_predicted_pd": g["pd_pred"].mean(),
            "avg_score": g["score"].mean(),
            "auc": safe_auc(g),
            "brier_score": brier_score_loss(g[TARGET_COL], g["pd_pred"]) if g[TARGET_COL].nunique() > 1 else np.nan,
        }))
        .reset_index()
    )
else:
    quarterly_performance = pd.DataFrame()

quarterly_performance

In [ ]:
if not quarterly_performance.empty:
    fig, ax = plt.subplots(figsize=(14, 5))

    for sample_name in quarterly_performance["sample"].unique():
        subset = quarterly_performance[quarterly_performance["sample"] == sample_name]
        ax.plot(
            subset["issue_quarter"].astype(str),
            subset["observed_bad_rate"],
            marker="o",
            linewidth=2,
            label=f"{sample_name} observed bad rate",
        )
        ax.plot(
            subset["issue_quarter"].astype(str),
            subset["avg_predicted_pd"],
            marker="x",
            linestyle="--",
            linewidth=2,
            label=f"{sample_name} avg predicted PD",
        )

    ax.set_xlabel("Issue quarter")
    ax.set_ylabel("Rate")
    ax.set_title("Observed Bad Rate and Predicted PD by Issue Quarter")
    ax.tick_params(axis="x", rotation=60)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / "bad_rate_pd_by_issue_quarter.png", dpi=150, bbox_inches="tight")
    plt.show()

## 13. Validation conclusion flags

These flags are a simple, transparent way to summarize model stability.


In [ ]:
oot_auc = performance_summary.loc[performance_summary["sample"] == "oot_test", "auc"].iloc[0]
train_auc = performance_summary.loc[performance_summary["sample"] == "train", "auc"].iloc[0]
oot_gini = performance_summary.loc[performance_summary["sample"] == "oot_test", "gini"].iloc[0]
oot_ks = performance_summary.loc[performance_summary["sample"] == "oot_test", "ks"].iloc[0]

score_psi_oot = score_psi_summary[
    (score_psi_summary["variable"] == "score")
    & (score_psi_summary["comparison_sample"] == "oot_test")
]["psi"].iloc[0]

max_feature_psi_oot = feature_psi_summary[
    feature_psi_summary["comparison_sample"] == "oot_test"
]["psi"].max()

oot_calibration_gap = calibration_summary[
    calibration_summary["sample"] == "oot_test"
]["calibration_gap"].iloc[0]

validation_flags = pd.DataFrame(
    [
        {
            "check": "OOT AUROC >= 0.60",
            "value": oot_auc,
            "status": "pass" if oot_auc >= 0.60 else "review",
        },
        {
            "check": "Train-to-OOT AUROC drop <= 0.05",
            "value": train_auc - oot_auc,
            "status": "pass" if (train_auc - oot_auc) <= 0.05 else "review",
        },
        {
            "check": "OOT Gini positive and meaningful",
            "value": oot_gini,
            "status": "pass" if oot_gini > 0.20 else "review",
        },
        {
            "check": "OOT KS positive and meaningful",
            "value": oot_ks,
            "status": "pass" if oot_ks > 0.15 else "review",
        },
        {
            "check": "OOT score PSI < 0.25",
            "value": score_psi_oot,
            "status": "pass" if score_psi_oot < 0.25 else "review",
        },
        {
            "check": "Max OOT feature PSI < 0.25",
            "value": max_feature_psi_oot,
            "status": "pass" if max_feature_psi_oot < 0.25 else "review",
        },
        {
            "check": "Absolute OOT calibration gap < 0.03",
            "value": abs(oot_calibration_gap),
            "status": "pass" if abs(oot_calibration_gap) < 0.03 else "review",
        },
    ]
)

validation_flags

## 14. Save validation outputs

In [ ]:
validation_artifacts = {
    "target_col": TARGET_COL,
    "model_features": model_features,
    "performance_summary": performance_summary,
    "performance_drift": performance_drift,
    "distribution_summary": distribution_summary,
    "score_psi_summary": score_psi_summary,
    "score_psi_detail": score_psi_detail,
    "feature_psi_summary": feature_psi_summary,
    "feature_psi_detail": feature_psi_detail,
    "calibration_summary": calibration_summary,
    "calibration_deciles": calibration_deciles,
    "quarterly_performance": quarterly_performance,
    "validation_flags": validation_flags,
}

with open(OUTPUT_DIR / "validation_and_stability_artifacts.pkl", "wb") as f:
    pickle.dump(validation_artifacts, f)

excel_path = OUTPUT_DIR / "validation_and_stability_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    performance_summary.to_excel(writer, sheet_name="performance_summary", index=False)
    performance_drift.to_excel(writer, sheet_name="performance_drift", index=False)
    distribution_summary.to_excel(writer, sheet_name="distribution_summary", index=False)
    score_psi_summary.to_excel(writer, sheet_name="score_psi_summary", index=False)
    score_psi_detail.to_excel(writer, sheet_name="score_psi_detail", index=False)
    feature_psi_summary.to_excel(writer, sheet_name="feature_psi_summary", index=False)
    feature_psi_detail.to_excel(writer, sheet_name="feature_psi_detail", index=False)
    calibration_summary.to_excel(writer, sheet_name="calibration_summary", index=False)
    calibration_deciles.to_excel(writer, sheet_name="calibration_deciles", index=False)
    quarterly_performance.to_excel(writer, sheet_name="quarterly_performance", index=False)
    validation_flags.to_excel(writer, sheet_name="validation_flags", index=False)

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'validation_and_stability_artifacts.pkl'}")
print(f"- {excel_path}")
print(f"- charts saved in {CHARTS_DIR}")

## 15. Conclusions

The validation and stability assessment indicates that the selected WOE Logistic Regression scorecard model demonstrates satisfactory discriminatory power, stability and population robustness across Train, Validation and Out-of-Time (OOT) samples.

## Performance Stability

The model achieved the following discrimination metrics:

| Sample     | AUROC | Gini  | KS    | Brier Score |
| ---------- | ----- | ----- | ----- | ----------- |
| Train      | 0.692 | 0.385 | 0.278 | 0.147       |
| Validation | 0.672 | 0.344 | 0.245 | 0.160       |
| OOT        | 0.667 | 0.335 | 0.246 | 0.136       |

The performance decline from Train to OOT remains limited:

* AUROC decrease: **2.5 percentage points**
* Gini decrease: **5.0 percentage points**
* KS decrease: **3.2 percentage points**

The model maintains stable discriminatory power across all samples and does not exhibit signs of material overfitting.

The OOT AUROC of 0.667 and OOT Gini of 0.335 indicate that the scorecard continues to separate good and bad accounts reasonably well on unseen future vintages.

## Population Stability

Population Stability Index (PSI) results are particularly strong.

### Score and PD Stability

| Variable     | Train vs Validation PSI | Train vs OOT PSI |
| ------------ | ----------------------- | ---------------- |
| Score        | 0.009                   | 0.014            |
| Predicted PD | 0.009                   | 0.014            |

All model-level PSI values remain substantially below the commonly used monitoring threshold of 0.10, indicating negligible population drift between development, validation and OOT samples.

From a portfolio perspective, the underlying population appears highly stable.

## Characteristic Stability

The characteristic-level PSI analysis shows that nearly all final model variables remain stable across Validation and OOT samples.

| Variable                 | OOT PSI |
| ------------------------ | ------- |
| revol_util_woe           | 0.344   |
| loan_to_income_woe       | 0.053   |
| dti_woe                  | 0.034   |
| acc_open_past_24mths_woe | 0.027   |
| purpose_woe              | 0.014   |
| mort_acc_woe             | 0.014   |
| Remaining variables      | < 0.01  |

One variable exceeds the commonly used monitoring threshold:

* `revol_util_woe` OOT PSI = **0.344**

This indicates a significant shift in the utilization distribution between development and OOT populations.

However:

* model-level PSI remains extremely low;
* model performance remains stable;
* calibration remains acceptable;
* no material deterioration in AUROC, Gini or KS is observed.

Therefore, this characteristic should be highlighted for ongoing monitoring after implementation, but it does not currently represent a material model risk.

## Calibration Assessment

Overall calibration remains reasonable across all samples.

| Sample     | Observed Bad Rate | Average Predicted PD | Calibration Gap |
| ---------- | ----------------- | -------------------- | --------------- |
| Train      | 19.95%            | 19.96%               | 0.01%           |
| Validation | 21.77%            | 18.86%               | -2.91%          |
| OOT        | 17.16%            | 18.80%               | +1.64%          |

The model slightly underestimates risk in the Validation sample and slightly overestimates risk in the OOT sample.

Nevertheless, all calibration gaps remain below 3 percentage points in absolute value, which is generally acceptable for an application scorecard at this stage of development.

Decile-level calibration analysis also confirms a consistent ranking of risk across all samples, with higher predicted PDs corresponding to higher observed bad rates.

## Vintage Stability

Quarterly performance analysis shows stable discriminatory power across origination vintages.

Observed bad rates fluctuate over time, particularly during the 2015–2017 period, reflecting changes in portfolio composition and risk environment.

Despite these variations:

* AUROC remains consistently within a relatively narrow range;
* average score remains stable;
* predicted PDs move in line with portfolio risk levels.

This behaviour suggests that the model captures underlying borrower risk rather than merely memorizing specific historical periods.

## Validation Checklist

The final validation review produced the following results:

| Validation Check                    | Result |
| ----------------------------------- | ------ |
| OOT AUROC ≥ 0.60                    | PASS   |
| Train-to-OOT AUROC drop ≤ 0.05      | PASS   |
| OOT Gini meaningful                 | PASS   |
| OOT KS meaningful                   | PASS   |
| OOT Score PSI < 0.25                | PASS   |
| Absolute OOT Calibration Gap < 0.03 | PASS   |
| Maximum OOT Feature PSI < 0.25      | REVIEW |

The only review item relates to `revol_util_woe`, which should be monitored through regular model monitoring and stability reporting.

## Overall Assessment

The validation exercise demonstrates that the final scorecard model is:

* statistically sound;
* stable across time;
* robust to future vintages;
* reasonably calibrated;
* resistant to material population drift;
* suitable for scorecard implementation and governance.

The model successfully satisfies the primary validation objectives of discrimination, stability, calibration and robustness.
